In [ ]:
import pandas as pd
import yaml
import numpy as np

In [ ]:
columns_file = "columns.yml"

# file path. Edit as needed.
metadata_file = "../../input_data/Metadata_file.csv"

In [ ]:
column_names = []

with open(columns_file) as stream:
    try:
        column_data = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

column_names = column_data['columns']
print(column_names)
print(len(column_names))

## Convert xlsx to Dataframe and extract the Papiri table

In [ ]:
#df = pd.read_excel("meta.xlsx", sheet_name="Papiri")
df = pd.read_csv(metadata_file)

In [ ]:
## Confirm
df.head(9)

## Remove the first X lines by extracting only the data rows (5 ~ end) and add Column names

In [ ]:
data_df = df.iloc[6:]
data_df.columns = column_names
data_df.shape

In [ ]:
data_df.head(3)

## Find rows and columns with no values whatsoever

In [ ]:
# Get rid of white space only cells (otherwise they are treated as holding values)
def replace_spaces_only(cell):
    if isinstance(cell, str) and cell.strip() == '':
        return None  # Replace with None, can be replaced with any value
    return cell

In [ ]:
data_df = data_df.map(replace_spaces_only)

In [ ]:
# Empty row indices 
nan_rows = data_df.isna().all(axis=1)
nan_row_indices = nan_rows[nan_rows].index
print(nan_row_indices)

In [ ]:
# Drop those rows
data_df = data_df.dropna(how='all')
data_df.shape

In [ ]:
# Confirm
nan_rows = data_df.isna().all(axis=1)
nan_row_indices = nan_rows[nan_rows].index
print(nan_row_indices)

In [ ]:
# Find empty column
drop_list = data_df.columns[data_df.isnull().all(0)].to_list()
print(drop_list)

In [ ]:
# clear leading and trailing spaces
data_df = data_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

## Clean up data

In [ ]:
# Convert -- to NaN
data_df = data_df.replace('--', np.nan)

In [ ]:
# reset index so that the first data row will be 0
data_df.reset_index(drop=True, inplace=True)
data_df

In [ ]:
# PapyrusNum forward fill 
data_df.loc[:, 'PapyrusNum'] = data_df.loc[:, 'PapyrusNum'].ffill()
data_df

In [ ]:
### If a row contains pezzo, fill in the CorniceNum 
### This works!  Consistent with the findings

for i in range(data_df.shape[0]):
    pezzo = data_df.at[i,'Pezzo']
    
    if not pd.isna(pezzo):
        data_df.at[i, 'CorniceNum'] = data_df.at[i-1,'CorniceNum']

data_df    

## Save it as a new csv file

In [ ]:
# import datetime

# today_str = datetime.datetime.today().strftime('%Y%m%d')
# data_df.to_csv(f'../../input_data/metadata_processed_{today_str}.csv', index=True)

## Investigate data

In [ ]:
def print_column_with_artifact(column_name):
        for _, row in data_df[data_df[column_name].notna()].iterrows():
            print(f"Papyrus: {row['PapyrusNum']}, Cornice: {row['CorniceNum']}, Pezzo: {row['Pezzo']}, \
            {column_name}: {row[f'{column_name}']}, ")
        

In [ ]:
def print_unique_values(column_name):
    print("Unique values: ", data_df[column_name].unique())

In [ ]:
def check_target_col(target_col, column_name):
    for _, row in data_df[data_df[column_name].notna()].iterrows():
        if pd.notna(row[target_col]):
            print(f"Papyrus: {row['PapyrusNum']}, Cornice: {row['CorniceNum']}, Pezzo: {row['Pezzo']}, \
            {column_name}: {row[f'{column_name}']}, ")

### Column: Unnamed

In [ ]:
print_column_with_artifact('Unnamed0')

### Column: Pezzo

In [ ]:
print_column_with_artifact('Pezzo')

### Column: Disegni

In [ ]:
print_column_with_artifact('Disegni')

### Column: UnrollingStatus

In [ ]:
#print_column_with_artifact('UnrollingStatus')
check_target_col('CorniceNum', 'UnrollingStatus') # Returns none -- that means everything is for Papyrus row

### Column: UUID

In [ ]:
# Find UUID assigned directory to Papyrus (or pezzi) rather than cornice
for _, row in data_df[data_df['UUID'].notna()].iterrows():
    if pd.isna(row['CorniceNum']):
        print(f"Papyrus: {row['PapyrusNum']}")

### Column: SupportMaterial

In [ ]:
# Find support material information for Papyrus row only 
for _, row in data_df[data_df['SupportMaterial'].notna()].iterrows():
    if pd.isna(row['CorniceNum']):
        print(f"Papyrus: {row['PapyrusNum']}")

### Column: Paperboard color

In [ ]:
# Find paperboard color where support material is missing
for _, row in data_df[data_df['PaperboardColor'].notna()].iterrows():
       if pd.isna(row['SupportMaterial']):
        print(f"Papyrus: {row['PapyrusNum']}")

In [ ]:
# Papyrus row only
for _, row in data_df[data_df['PaperboardColor'].notna()].iterrows():
    if not row['CorniceNum']:
        print(f"Papyrus: {row['PapyrusNum']}")

### Column: OGStorageNum

In [ ]:
print_column_with_artifact('OGStorageNum')

### Column: PreviouslyKnownAs

In [ ]:
print_column_with_artifact('PreviouslyKnownAs')

### Column: CurrentlyKnownAs

In [ ]:
print_column_with_artifact('CurrentlyKnownAs')

### Column: SeeAlsoPHerc

In [ ]:
print_column_with_artifact('SeeAlsoPHerc')

### Column: AlternativeUUID

In [ ]:
print_column_with_artifact('AlternativeUUID') # none currently

### Column: CustodialInstitution

In [ ]:
#print_column_with_artifact('CustodialInstitution')   # too many
check_target_col('CorniceNum', 'CustodialInstitution') # None -> All are papyrus

### Columns: CustodialURI, CustodialCityCountry, CityCountryURI

Should be straightforward

### Column: StorageLocation

In [ ]:
#print_column_with_artifact('StorageLocation')
check_target_col('Pezzo', 'StorageLocation')

### Column: StorageLocationNotUnrolledPortion

In [ ]:
#print_column_with_artifact("StorageLocationNotUnrolledPortion")

### Column: History

In [ ]:
print_column_with_artifact("History")

### Column: CustodialHistory

In [ ]:
print_column_with_artifact("CustodialHistory")

### Columns: ObjectFormat, ObjectFormatURI

In [ ]:
#print_column_with_artifact('ObjectFormat')
check_target_col('CorniceNum', 'ObjectFormat') # returns none --> all are papyrus

### Columns: MaterialType, MaterialTypeURI

In [ ]:
#print_column_with_artifact('MaterialType')
check_target_col('CorniceNum', 'MaterialType') # returns none --> all are papyrus

### Column: Diameter

In [ ]:
#print_column_with_artifact('Diameter')
check_target_col('CorniceNum', 'Diameter') # returns none --> all are papyrus

### Column: Height

In [ ]:
check_target_col('CorniceNum', 'Height')
# print_column_with_artifact('Height')  # returns many

### Column: Width

In [ ]:
check_target_col('CorniceNum', 'Width')
print_column_with_artifact('Width')

## Column: Grams

In [ ]:
#print_column_with_artifact("Grams")
check_target_col("CorniceNum", "Grams") # returns none -- all papyrus

### Columns: Language, Language URI

Language URI should be a property

In [ ]:
check_target_col("CorniceNum", "Language")
print_unique_values("Language")

### Column: UnrollerPerson and UnrollingMethod

In [ ]:
print_column_with_artifact("UnrollerPerson")
print_unique_values("UnrollerPerson")

In [ ]:
### Column: UnrollingMethod

In [ ]:
print_unique_values("UnrollingMethod")

### Column: UnrolledDate

In [ ]:
print_column_with_artifact("UnrolledDate")

### Column: UnrolledBeforeDate 

In [ ]:
print_column_with_artifact("UnrolledBeforeDate")

### Column: UnrolledAfterDate

In [ ]:
print_column_with_artifact("UnrolledAfterDate")

### Columns: OsloMethod and Scorze1

In [ ]:
print_column_with_artifact("OsloMethod")
print_unique_values("OsloMethod")

In [ ]:
# print_column_with_artifact("Scorze1")

### Column: Note

In [ ]:
print_column_with_artifact("Note")

### Columns: PhotographFiles, OtherPhotographs

In [ ]:
# print_column_with_artifact("PhotographFiles")

In [ ]:
check_target_col("CorniceNum", "PhotographFiles")  # No Cornice

In [ ]:
print_column_with_artifact("OtherPhotographs")

### Column: BibliographyLink

In [ ]:
print_column_with_artifact("BibliographyLink")

In [ ]:
check_target_col("CorniceNum", "BibliographyLink")

### Column: Editions

In [ ]:
print_column_with_artifact("Editions")

In [ ]:
check_target_col("CorniceNum", "Editions")

### Column: FUrtherBibliography

In [ ]:
print_column_with_artifact("FurtherBibliography")

In [ ]:
check_target_col("CorniceNum", "FurtherBibliography")

### Columns TrismegistosNum, TrismegistosURI

In [ ]:
print_column_with_artifact("TrismegistosNum")

In [ ]:
print_column_with_artifact("TrismegistosURI")

### Columns: LdabNum and LdabURI

In [ ]:
print_column_with_artifact("LdabNum")

In [ ]:
print_column_with_artifact("LdabURI")

### Columns: DclpURI and DclpScrollsURI

In [ ]:
print_column_with_artifact("DclpURI")

In [ ]:
print_column_with_artifact("DclpScrollsURI")

### Column: Scorze (AZ)

In [ ]:
# print_column_with_artifact("Scorze") 

In [ ]:
check_target_col("CorniceNum", "Scorze")  # None = All are papyrus

### Columns: Cornici, CorniciCount, PezziCount

In [ ]:
#print_column_with_artifact("Cornici")
print_unique_values("Cornici")

In [ ]:
check_target_col("CorniceNum", "Cornici")

In [ ]:
#print_column_with_artifact("CorniciCount")
print_unique_values("CorniciCount")

In [ ]:
check_target_col("CorniceNum", "CorniciCount")

In [ ]:
#print_column_with_artifact("PezziCount")
print_unique_values("PezziCount")

### Column: Author

In [ ]:
print_unique_values("Author")

### Column: LiteraryWork

In [ ]:
print_unique_values("LiteraryWork")

### Column: Subscriptio, SubscriptioLocation

In [ ]:
print_unique_values("Subscriptio")

In [ ]:
print_unique_values("SubscriptioLocation")

### Columns: InitialEndTitle, RectoVersoTitle

In [ ]:
print_unique_values("InitialEndTitle")

In [ ]:
print_unique_values("RectoVersoTitle")

### Column: StoredWithPHercNum

In [ ]:
print_column_with_artifact("StoredWithPHercNum")

### Column: Engravings

In [ ]:
print_column_with_artifact("Engravings")

### Column: Transcripts

In [ ]:
print_column_with_artifact("Transcripts")

### Column: CavalloScribalStyle

In [ ]:
print_column_with_artifact("CavalloScribalStyle")

### Column: MultipleHands

In [ ]:
print_column_with_artifact("MultipleHands")

### Column: NeapolitanDrawings, NeapolitanDrawings2, NeapolitanDrawings3

Are these different?  Apparently not.

In [ ]:
for _, row in data_df[data_df['NeapolitanDrawings'].notna()].iterrows():
    if (row['NeapolitanDrawings2'] != row['NeapolitanDrawings']) | (row['NeapolitanDrawings3'] != row['NeapolitanDrawings']):
        print(f"{row['NeapolitanDrawings']}, {row['NeapolitanDrawings2']}, {row['NeapolitanDrawings3']}")

# Returns none -- these columns are the same

In [ ]:
# #Sanity check 
# for _, row in data_df[data_df['NeapolitanDrawings'].notna()].iterrows():
#     print(f"{row['PapyrusNum']}: {row['NeapolitanDrawings']}, {row['NeapolitanDrawings2']}, {row['NeapolitanDrawings3']}")

In [ ]:
print_column_with_artifact('NeapolitanDrawings')

### Columns: OxonianDrawings, OxonianDrawings2, OxonianDrawings3

Are these different? --> The only difference is Pherc 238, where 3rd column is missing the value

In [ ]:
for _, row in data_df[data_df['OxonianDrawings'].notna()].iterrows():
    if (row['OxonianDrawings2'] != row['OxonianDrawings']) | (row['OxonianDrawings3'] != row['OxonianDrawings']):
        print(f"{row['PapyrusNum']}: {row['OxonianDrawings']}, {row['OxonianDrawings2']}, {row['OxonianDrawings3']}")

In [ ]:
# for _, row in data_df[data_df['OxonianDrawings'].notna()].iterrows():
#     print(f"{row['PapyrusNum']}: {row['OxonianDrawings']}, {row['OxonianDrawings2']}, {row['OxonianDrawings3']}")

In [ ]:
print_column_with_artifact('OxonianDrawings')

### Column: OtherSameScrollPapyri

In [ ]:
print_column_with_artifact("OtherSameScrollPapyri")

### Column: AdditionalNotes

In [ ]:
print_column_with_artifact("AdditionalNotes")